# ESCM 2.0: self-excitation without scheduler tricks

## Question and numerical contract

Notebook 12 showed exact Brian2–NumPy agreement on elementary microcircuits, but an **additive autapse without refractory** exhibited a firing rate that grew as the timestep shrank. This notebook distinguishes the mathematical mechanism from its simulator implementation.

We compare five autaptic rules: (A) assignment \(I_E\leftarrow w\), as in original ESCM; (B) addition \(I_E\leftarrow I_E+w\); (C) addition plus 2-ms absolute refractory; (D) saturation \(I_E\leftarrow I_E+w(1-I_E/I_{max})\); and (E) saturation plus refractory. Saturation is phenomenological, not a validated model of short-term depression.

All five use a standard LIF membrane, exponential synaptic currents, an ordinary self-synapse, and **no custom events or manual scheduling**. This test does *not* reproduce the original ESCM persistence gate.

\[
\tau_m\dot v=-v+I_{ext}+I_E-I_I,\qquad\tau_x\dot I_x=-I_x.
\]

We use \(\tau_m=10\) ms, \(\tau_{ext}=3\) ms, \(\tau_E=5\) ms, \(\tau_I=12\) ms, threshold 1, external impulse 6 at 1 ms, and an optional inhibitory impulse 12 at 35 ms. The parameters are an exploratory numerical test, **not physiological estimates**. Different decay constants and descriptive names avoid the symbolic collisions found in notebook 12.

Evidence required: autonomous spiking during 20–35 ms; reasonable late firing rate as the timestep shrinks; reduced activity after inhibition compared with a matched unperturbed run. An imposed inhibitory pulse tests susceptibility, not naturally emerging lateral competition.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from brian2 import *

prefs.codegen.target = "numpy"
tm, tx, te, ti = 10*ms, 3*ms, 5*ms, 12*ms
vth, input_gain, inhibition_gain, cap = 1.0, 6.0, 12.0, 24.0
duration_ms = 80.0
weights = (4.0, 8.0, 12.0)
timesteps = (0.5, 0.25, 0.1, 0.05, 0.025)
variants = {
    "assignment": ("assign", 0.0),
    "additive": ("add", 0.0),
    "additive_refractory": ("add", 2.0),
    "saturating": ("saturate", 0.0),
    "saturating_refractory": ("saturate", 2.0),
}
print(f"{len(variants)*len(weights)*len(timesteps)} timestep runs, plus inhibitory perturbations")

## Single-neuron experiment

The input is delivered through a standard SpikeGeneratorGroup, and a separate optional source delivers a 35-ms inhibitory event. To avoid a previously encountered Brian2 error, only actually used synapses are created. Synaptic variables are explicitly initialized; the state-monitor trace is collected only for selected diagnostic runs.

Assignment and additive updates are different **models**, not equivalent implementations. The refractory period is also a genuine model change.

In [ ]:
def simulate(model, weight, step_ms, inhibit=False, record=False):
    start_scope()
    defaultclock.dt = step_ms*ms
    kind, refractory_ms = variants[model]
    params = dict(tm_model=tm, tx_model=tx, te_model=te, ti_model=ti,
                  vth_model=vth, input_gain=input_gain, inhibition_gain=inhibition_gain,
                  self_weight=weight, cap_model=cap)
    eq = """
    dv/dt = (-v + Iext + Iexc - Iinh)/tm_model : 1
    dIext/dt = -Iext/tx_model : 1
    dIexc/dt = -Iexc/te_model : 1
    dIinh/dt = -Iinh/ti_model : 1
    """
    neuron = NeuronGroup(1, eq, threshold="v > vth_model",
                         reset="v = 0", refractory=refractory_ms*ms,
                         method="exact", namespace=params)
    neuron.v = neuron.Iext = neuron.Iexc = neuron.Iinh = 0
    camera = SpikeGeneratorGroup(1, [0], [1.0]*ms)
    feed = Synapses(camera, neuron, on_pre="Iext_post += input_gain", namespace=params)
    feed.connect(j="i")
    rule = ("Iexc_post = self_weight" if kind == "assign" else
            "Iexc_post += self_weight" if kind == "add" else
            "Iexc_post += self_weight*(1 - Iexc_post/cap_model)")
    self_syn = Synapses(neuron, neuron, on_pre=rule, namespace=params)
    self_syn.connect(i=0, j=0)
    spikes = SpikeMonitor(neuron)
    objects = [camera, neuron, feed, self_syn, spikes]
    if inhibit:
        source = SpikeGeneratorGroup(1, [0], [35.0]*ms)
        syn = Synapses(source, neuron, on_pre="Iinh_post += inhibition_gain",
                       namespace=params)
        syn.connect(j="i")
        objects += [source, syn]
    if record:
        state = StateMonitor(neuron, ["v", "Iexc", "Iinh"], record=True, when="end")
        objects.append(state)
    Network(*objects).run(duration_ms*ms)
    result = {"spikes": np.asarray(spikes.t/ms)}
    if record:
        result["t"] = np.asarray(state.t/ms)
        result["states"] = {name: np.asarray(getattr(state, name)[0]).copy()
                            for name in ("v", "Iexc", "Iinh")}
        assert all(np.isfinite(x).all() for x in result["states"].values())
    return result

## Timestep convergence

Run all five models for three autaptic weights and five timesteps. Late firing is measured over **50–80 ms** (30 ms), well after the external impulse. The fraction change between the two finest steps is descriptive evidence of convergence, not proof. A high rate scaling with \(1/\Delta t\) is a warning that spiking may be timestep-limited.

**A model that does not persist is not successful merely because its firing rate converges to zero.**

In [ ]:
rows = []
for model in tqdm(variants, desc="Autapse variants"):
    for weight in weights:
        for dt_ms in timesteps:
            spikes = simulate(model, weight, dt_ms)["spikes"]
            rows.append(dict(model=model, weight=weight, dt_ms=dt_ms,
                total_spikes=len(spikes),
                persistent_spikes=np.count_nonzero((spikes >= 20) & (spikes < 35)),
                late_rate_hz=np.count_nonzero((spikes >= 50) & (spikes < 80))/0.030))
results = pd.DataFrame(rows)
display(results)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
for ax, weight in zip(axes, (8.0, 12.0)):
    for model in variants:
        subset = results[(results.model == model) & (results.weight == weight)]
        ax.plot(subset.dt_ms, subset.late_rate_hz, marker="o", label=model)
    ax.set(xscale="log", xlabel="Simulation timestep (ms)",
           ylabel="Late firing rate (Hz)", title=f"Autaptic weight = {weight:g}")
    ax.grid(alpha=0.2)
axes[0].legend(fontsize=8)
plt.show()
print("Relative change in late rate: 0.05 ms versus 0.025 ms")
for (model, weight), g in results.groupby(["model", "weight"], sort=False):
    x = g.set_index("dt_ms").late_rate_hz
    print(f"{model:24s} w={weight:g}: {abs(x[0.025]-x[0.05])/max(x[0.025],x[0.05],1):.1%}")

## Inhibitory interruption: matched control versus perturbation

Repeat each configuration at 0.05 ms, with and without an inhibitory event at 35 ms. The comparison distinguishes a transient pause in firing (35–50 ms) from loss or recovery of persistence later (60–80 ms). The inhibitory event is **externally supplied**; it does not demonstrate a functional inhibitory microcircuit.

A model is a promising candidate only when it shows both autonomous activity *and* measurable inhibition without unreasonable timestep dependence.

In [ ]:
rows = []
for model in tqdm(variants, desc="Inhibitory perturbation"):
    for weight in weights:
        control = simulate(model, weight, 0.05)["spikes"]
        inhibited = simulate(model, weight, 0.05, inhibit=True)["spikes"]
        count = lambda s, a, b: int(np.count_nonzero((s >= a) & (s < b)))
        rows.append(dict(model=model, weight=weight,
            before=count(control,20,35),
            control_35_50=count(control,35,50),
            inhibited_35_50=count(inhibited,35,50),
            inhibited_60_80=count(inhibited,60,80)))
inhibition = pd.DataFrame(rows)
display(inhibition)
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True, constrained_layout=True)
for ax, model in zip(axes, ("assignment", "additive_refractory", "saturating_refractory")):
    for inhibited, label in ((False, "Control"), (True, "Inhibitory pulse")):
        r = simulate(model, 8.0, 0.05, inhibit=inhibited, record=True)
        ax.plot(r["t"], r["states"]["Iexc"], label=f"{label}: Iexc")
        ax.plot(r["t"], r["states"]["Iinh"], linestyle=":", label=f"{label}: Iinh")
    ax.axvline(35, linestyle="--", color="black")
    ax.set(ylabel="Current (model units)", title=model)
    ax.grid(alpha=0.2)
axes[0].legend(ncol=2, fontsize=8)
axes[-1].set_xlabel("Time (ms)")
plt.show()

## Interpretation

**If assignment passes:** it suggests bounded current refresh can support persistence without a custom \(P\) gate, but it is still a phenomenological update. **If refractory additive passes:** inspect the current traces to ensure the refractory is not merely hiding unbounded accumulating excitation. **If saturation passes:** it provides a bounded-current candidate that could be connected to a richer short-term-plasticity model in subsequent work.

This test is **isolated-neuron-only**. It does not establish spatial selectivity or the suppression of obsolete moving edges. Do not change the original ESCM or optimize spatial kernels on the strength of this notebook alone.

**Status:** This notebook must be executed in a Brian2 environment to produce empirical results; the formulations are hypotheses, not findings.